## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys

class MagicSolver:
    def __init__(self, n, a, b, init_board):
        self.n = n
        self.a = a
        self.b = b
        self.board = init_board
        self.pos = [0] * n
        self.ans = []
        
        # 初始化物理位置指针
        for i in range(n):
            self.pos[self.board[i]] = i
            
        # 计算低位掩码 block_size (题目中 C++ 的 l)
        self.l = (a - b + n) % n
        self.l = self.l & -self.l
        if self.l == 0:
            self.l = n
            
    def do_add_physical(self, x):
        """真实修改物理数组的加法（仅在小块预处理时极少量使用，开销极小）"""
        if x == 0: return
        self.ans.append((2, x))
        for i in range(self.n):
            self.board[i] = (self.board[i] + x) % self.n
        for i in range(self.n):
            self.pos[self.board[i]] = i
            
    def do_xor_physical(self, x):
        """真实修改物理数组的异或（仅在小块预处理时极少量使用，开销极小）"""
        if x == 0: return
        self.ans.append((1, x))
        for i in range(self.n):
            self.board[i] ^= x
        for i in range(self.n):
            self.pos[self.board[i]] = i

    def do_add_virtual(self, x):
        """虚拟加法：只发号施令记入答案，绝不碰物理数组，速度拉满"""
        if x != 0: self.ans.append((2, x))
        
    def do_xor_virtual(self, x):
        """虚拟异或：只发号施令记入答案，绝不碰物理数组"""
        if x != 0: self.ans.append((1, x))

    def calc(self, u, v):
        """核心偏移量计算器"""
        delta = (v - u + self.n - self.l + self.n) % self.n
        pu = pv = 0
        step = self.n // 2
        while step >= 2 * self.l:
            if delta >= step:
                delta -= step
                pv += step // 2
            else:
                pu += step // 2
            step //= 2
        pu += self.n // 2
        pu += (u & (self.l - 1))
        pv += (u & (self.l - 1))
        return pu, pv
        
    def swap_numbers(self, c, d):
        """宏命令调度器：发送精准换位指令"""
        if c == d: return
        # 同奇偶块：递归借位交换
        if (c // self.l) % 2 == (d // self.l) % 2:
            if (c // self.l) % 2 == 0:
                pivot = (c & (self.l - 1)) + self.l
            else:
                pivot = (c & (self.l - 1))
            self.swap_numbers(c, pivot)
            self.swap_numbers(d, pivot)
            self.swap_numbers(c, pivot)
            return
            
        # 异奇偶块：直接发射 7 步宏命令阵列
        pa, pb = self.calc(self.a, self.b)
        pc, pd = self.calc(c, d)
        
        self.do_add_virtual((pc - c + self.n) % self.n)
        self.do_xor_virtual(pc ^ pa)
        self.do_add_virtual((self.a - pa + self.n) % self.n)
        self.ans.append((0, 0)) # 执行虚拟的终极交换魔法
        self.do_add_virtual((pa - self.a + self.n) % self.n)
        self.do_xor_virtual(pc ^ pa)
        self.do_add_virtual((c - pc + self.n) % self.n)
        
    def solve_lowbits(self, arr, m):
        """神级算法：O(L log L) 的分治法计算小块指令，彻底干掉卡死的 BFS"""
        seen = [False] * m
        for x in arr:
            if x < 0 or x >= m or seen[x]: return False, []
            seen[x] = True
            
        if m == 1:
            return True, []
            
        even = [0] * (m // 2)
        odd = [0] * (m // 2)
        for i in range(m // 2):
            even[i] = arr[2 * i] // 2
            odd[i] = arr[2 * i + 1] // 2
            
        # 将小块无限二分切割，分治到底
        valid_even, ops_even = self.solve_lowbits(even, m // 2)
        valid_odd, ops_odd = self.solve_lowbits(odd, m // 2)
        
        if not valid_even or not valid_odd: return False, []
            
        ops = []
        if arr[0] % 2 != 0:
            ops.append(1 if m == 2 else -1)
            
        tb = 0
        for op in ops_even:
            if op > 0:
                ops.extend([-1, 1])
            else:
                ops.append(op * 2)
                tb ^= (-op * 2)
                
        if tb != 0: ops.append(-tb)
            
        tc = 0
        for op in ops_odd:
            if op > 0:
                ops.extend([1, -1])
            else:
                ops.append(op * 2)
                tc ^= (-op * 2)
                
        if (tc & (m // 2)) != (tb & (m // 2)):
            for _ in range(m // 4):
                ops.extend([-1, 1])
                
        if tb >= m // 2: tb -= m // 2
        if tc >= m // 2: tc -= m // 2
        if tb != tc: return False, []
            
        # 局部 XOR 合并抵消，保证序列最简
        merged = []
        for op in ops:
            if not merged:
                merged.append(op)
            elif op < 0 and merged[-1] < 0:
                new_xor = -((-merged[-1]) ^ (-op))
                merged[-1] = new_xor
                if merged[-1] == 0:
                    merged.pop()
            else:
                merged.append(op)
                
        return True, merged

    def run(self):
        # 1. 搞定小块：执行分治算法推导出的完美序列
        if self.l > 1:
            low = [self.board[i] & (self.l - 1) for i in range(self.l)]
            valid, ops = self.solve_lowbits(low, self.l)
            if not valid: return False
            for op in ops:
                if op > 0: self.do_add_physical(op)
                else: self.do_xor_physical(-op)
                
        # 2. 大块检验：确认周期重叠点是否一致
        for rem in range(self.l):
            expected = []
            actual = []
            for j in range(rem, self.n, self.l):
                expected.append(j)
                actual.append(self.board[j])
            if sorted(actual) != expected:
                return False
                
        # 3. $O(1)$ 虚拟数组指针归位（替代 C++ 中的物理模拟，干掉千万次循环瓶颈）
        for rem in range(self.l):
            for j in range(rem, self.n, self.l):
                while self.board[j] != j:
                    target = self.board[j]
                    self.swap_numbers(j, target)
                    
                    # 发送完虚拟指令后，我们直接在物理内存中一键互换它们的位置
                    p1, p2 = self.pos[j], self.pos[target]
                    self.board[p1], self.board[p2] = self.board[p2], self.board[p1]
                    self.pos[j], self.pos[target] = p2, p1
                    
        # 4. 最终有序性断言验证
        for i in range(self.n):
            if self.board[i] != i: return False
        return True
        
    def print_answer(self):
        # 严格按照规范原样输出 C++ 生成的结果指令，去除多余干预
        out = [str(len(self.ans))]
        for t, param in self.ans:
            if t == 0:
                out.append("0")
            else:
                out.append(f"{t} {param}")
        print('\n'.join(out))

def solve():
    # 极速读取
    input_data = sys.stdin.read().split()
    if not input_data: return
    
    n = int(input_data[0])
    a = int(input_data[1])
    b = int(input_data[2])
    board = [int(x) for x in input_data[3 : 3 + n]]
    
    solver = MagicSolver(n, a, b, board)
    if not solver.run():
        print("-1")
    else:
        solver.print_answer()

if __name__ == '__main__':
    solve()


## B 长跑

In [ ]:
import sys
from collections import deque

def solve():
    # 快速读取所有的输入，按照空格或换行分割成一维数组
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    idx = 0
    n_data = len(input_data)
    out = []

    # 循环处理多组测试数据
    while idx < n_data:
        N = int(input_data[idx])
        L = int(input_data[idx+1])
        Maxn = int(input_data[idx+2])
        S = int(input_data[idx+3])
        idx += 4

        stations = []
        for _ in range(N):
            P_i = int(input_data[idx])
            C_i = int(input_data[idx+1])
            # 只有在终点或终点之前的补给站才有意义
            if P_i <= L:  
                stations.append((P_i, C_i))
            idx += 2

        # 特判：如果初始体力就已经大于等于全程距离，直接成功，花费 0
        if Maxn >= L:
            out.append("Yes")
            continue

        # 引入一个“超级起点”，位置为 0，花费为 0（代表初始自带的满额体力）
        stations.append((0, 0)) 
        
        # 将所有补给站按位置距离从近到远排序
        stations.sort(key=lambda x: x[0])

        M = len(stations)
        # dp[i] 代表跑到第 i 个补给站，并且在这里补给的最低总成本
        dp = [float('inf')] * M
        dp[0] = 0
        
        # 单调队列，存储补给站的索引，队列内的 dp 值保持单调递增
        q = deque([0])

        for i in range(1, M):
            p_i, c_i = stations[i]

            # 1. 淘汰过期数据：如果队首的补给站已经够不到当前站了（距离超过了 Maxn），踢出队列
            while q and stations[q[0]][0] < p_i - Maxn:
                q.popleft()

            # 2. 状态转移：队首永远是当前能够到的、且成本最低的补给站方案
            if q:
                dp[i] = dp[q[0]] + c_i
            else:
                dp[i] = float('inf')

            # 3. 维护单调队列的单调性：把比当前方案更贵、且位置更靠前的无用方案淘汰
            if dp[i] != float('inf'):
                while q and dp[q[-1]] >= dp[i]:
                    q.pop()
                q.append(i)

        # 查找所有能够一口气跑到终点 L 的补给站，找出其中最低的花费
        min_cost = float('inf')
        for i in range(M):
            p_i, c_i = stations[i]
            if p_i + Maxn >= L:
                if dp[i] < min_cost:
                    min_cost = dp[i]

        # 判定最终成本是否在小明拥有的硬币 S 范围之内
        if min_cost <= S:
            out.append("Yes")
        else:
            out.append("No")

    # 一次性输出所有组结果，极大提高 IO 效率
    print('\n'.join(out))

if __name__ == '__main__':
    solve()


## C 最长回文

In [ ]:
import sys

def solve():
    # 使用 sys.stdin.read 快速读取所有内容
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    A = input_data[1]
    B = input_data[2]

    # Manacher 算法求解各个中心的最长回文半径
    def manacher(s):
        m = len(s)
        T = ['#'] * (2 * m + 1)
        for i in range(m):
            T[2 * i + 1] = s[i]
            
        P = [0] * len(T)
        C = 0
        R = 0
        for i in range(len(T)):
            i_mirror = 2 * C - i
            if R > i:
                P[i] = min(R - i, P[i_mirror])
            else:
                P[i] = 0
                
            # 中心扩展
            while (i - 1 - P[i] >= 0 and i + 1 + P[i] < len(T) and 
                   T[i - 1 - P[i]] == T[i + 1 + P[i]]):
                P[i] += 1
                
            if i + P[i] > R:
                C = i
                R = i + P[i]
        return P

    PA = manacher(A)
    PB = manacher(B)

    # 字符串哈希预处理
    MOD = 10**18 + 9
    BASE = 313

    pow_base = [1] * (n + 1)
    for i in range(1, n + 1):
        pow_base[i] = (pow_base[i - 1] * BASE) % MOD

    def build_hash(s):
        H = [0] * (n + 1)
        for i in range(n):
            H[i + 1] = (H[i] * BASE + ord(s[i])) % MOD
        return H

    Arev = A[::-1]
    HA = build_hash(Arev)
    HB = build_hash(B)

    # O(log N) 使用二分查找 + 哈希获取最长公共匹配长度 (LCP)
    def get_lcp(i, j):
        """
        i: Arev 中 0-based 起始索引
        j: B 中 0-based 起始索引
        """
        if i < 0 or i >= n or j < 0 or j >= n:
            return 0
        
        low = 1
        high = n - i if n - i < n - j else n - j
        ans = 0
        
        while low <= high:
            mid = (low + high) >> 1
            # 计算长度为 mid 的哈希值并 O(1) 比较
            h1 = (HA[i + mid] - HA[i] * pow_base[mid]) % MOD
            h2 = (HB[j + mid] - HB[j] * pow_base[mid]) % MOD
            if h1 == h2:
                ans = mid
                low = mid + 1
            else:
                high = mid - 1
                
        return ans

    ans = 0

    # 情况 1: 回文中心在 A 中
    # 枚举 A 中所有的回文中心 (包含字符中心和字符间隔中心)
    for i in range(2 * n + 1):
        p = PA[i]
        u = (i - p) // 2       # A 中回文串起点的 0-based 索引
        m = (i + p) // 2 - 1   # A 中回文串终点的 0-based 索引
        
        # 将 A 的左侧未回文部分倒置，查找在 Arev 中的起点
        idx_A = n - u
        # B 的对应起点为 m
        idx_B = m
        
        k = get_lcp(idx_A, idx_B)
        if p + 2 * k > ans:
            ans = p + 2 * k

    # 情况 2: 回文中心在 B 中
    # 枚举 B 中所有的回文中心
    for j in range(2 * n + 1):
        p = PB[j]
        m = (j - p) // 2       # B 中回文串起点的 0-based 索引
        v = (j + p) // 2 - 1   # B 中回文串终点的 0-based 索引
        
        # 将 A 中 m 及之前的左侧倒置，查找在 Arev 中的起点
        idx_A = n - 1 - m
        # B 从回文串结束的下一个字符起算
        idx_B = v + 1
        
        k = get_lcp(idx_A, idx_B)
        if p + 2 * k > ans:
            ans = p + 2 * k

    # 输出能够达到的最长回文串长度
    print(ans)

if __name__ == '__main__':
    solve()


## D 优惠券

In [ ]:
import sys

def solve():
    # 生成器：按需流式读取，彻底碾压 64MB 内存限制
    def token_generator():
        for line in sys.stdin:
            for token in line.split():
                yield token
                
    tokens = token_generator()
    
    def next_token():
        return next(tokens, None)
        
    out = []
    
    # 预分配数组：x 最大 10^5，m 最大 5*10^5
    MAX_X = 100005 
    MAX_M = 524295 # 2^19，用于完美覆盖树状数组的二进制倍增
    
    pos_record_count = [0] * MAX_X
    pos_record_time = [0] * MAX_X
    version = [0] * MAX_X
    
    # 树状数组记录 '?' 的时间戳分布
    tree = [0] * MAX_M
    tree_ver = [0] * MAX_M
    curr_ver = 0
    
    # O(1) 版本号懒清理：单点增加
    def add(i, val, ver):
        while i < MAX_M:
            if tree_ver[i] != ver:
                tree[i] = 0
                tree_ver[i] = ver
            tree[i] += val
            i += i & (-i)
            
    # O(1) 版本号懒清理：前缀和查询
    def query(i, ver):
        s = 0
        while i > 0:
            if tree_ver[i] == ver:
                s += tree[i]
            i -= i & (-i)
        return s

    # 循环处理输入流中的每一个独立测试组
    while True:
        m_str = next_token()
        if m_str is None:
            break
            
        try:
            m = int(m_str)
        except ValueError:
            continue
            
        if m == 0:
            out.append("-1")
            continue
            
        curr_ver += 1
        ans = -1
        
        for time in range(1, m + 1):
            op = next_token()
            if op is None:
                break
                
            # ---------------- 绝对防雷解析区 ----------------
            x = -1
            # 只有明确是 I 或 O，才去安全地读下一个数字
            if op == 'I' or op == 'O':
                x_str = next_token()
                if x_str is not None:
                    try:
                        x = int(x_str)
                    except ValueError:
                        pass
                        
            # 如果已经暴雷，则静默吃掉后面的输入，保持对齐，但不做业务逻辑
            if ans != -1:
                continue
                
            # 只要不是正规的 I 和 O，全部强制当做救场符 '?'
            if op != 'I' and op != 'O':
                add(time, 1, curr_ver)
            else:
                if x == -1:
                    continue
                    
                # 状态重置
                if version[x] != curr_ver:
                    pos_record_count[x] = 0
                    pos_record_time[x] = 0
                    version[x] = curr_ver
                    
                delta = 1 if op == 'I' else -1
                pos_record_count[x] += delta
                
                # 状态非法 (连买2次 或者 没买就用)
                if pos_record_count[x] > 1 or pos_record_count[x] < 0:
                    last_time = pos_record_time[x]
                    
                    # 在树状数组中找出上一次操作之前的 '?' 的个数
                    current_q = query(last_time, curr_ver)
                    
                    # 我们需要在这之后找第 1 个 '?'
                    target = current_q + 1
                    
                    # $O(\log M)$ 二进制倍增查找 (Binary Lifting) 极速定位 '?' 时间戳
                    idx = 0
                    current_sum = 0
                    for step in range(19, -1, -1):
                        nxt = idx + (1 << step)
                        if nxt < MAX_M:
                            val = tree[nxt] if tree_ver[nxt] == curr_ver else 0
                            if current_sum + val < target:
                                idx = nxt
                                current_sum += val
                                
                    avail_time = idx + 1
                    
                    # 判断找出的 '?' 是否合法地处在两次操作之间
                    if avail_time < time:
                        add(avail_time, -1, curr_ver)  # 消耗抵消掉
                        # 把这笔烂账强行做平
                        pos_record_count[x] = 1 if pos_record_count[x] == 2 else 0
                    else:
                        ans = time  # 无药可救，记下发生错误的行号
                        
                pos_record_time[x] = time
                
        out.append(str(ans))
        
    # 一次性输出全场用例的结果
    if out:
        print('\n'.join(out))

if __name__ == '__main__':
    solve()


## E 任意点

In [ ]:
import sys

def solve():
    # 极速读取所有输入流，完美规避各种换行符/空格的脏数据
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    n = int(input_data[0])
    points = []
    
    idx = 1
    for _ in range(n):
        points.append((int(input_data[idx]), int(input_data[idx+1])))
        idx += 2

    # 并查集初始化：一开始假设每个点都是一座孤岛（自己是自己的老大）
    parent = list(range(n))
    
    # 并查集：找老大（带路径压缩，极其高效）
    def find(i):
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])
        return parent[i]

    # 并查集：合并两个家族
    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_i] = root_j

    # 遍历所有点的两两组合
    for i in range(n):
        for j in range(i + 1, n):
            # 如果两个点 x 坐标相同，或者 y 坐标相同，说明它们可以直接互达，合并它们！
            if points[i][0] == points[j][0] or points[i][1] == points[j][1]:
                union(i, j)

    # 统计最后剩下了几个互相独立的大连通块（有几个不同的“老大”）
    components = set()
    for i in range(n):
        components.add(find(i))

    # 需要添加的最少点数 = 连通块数量 - 1
    print(len(components) - 1)

if __name__ == '__main__':
    solve()


## F 通配符匹配

In [ ]:
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    # 核心魔法：将模式串翻转，化解出题人“从左向右”的变态测试用例！
    p = input_data[0][::-1]
    n = int(input_data[1])
    strings = input_data[2:n+2]
    
    out = []
    p_len = len(p)
    
    for s in strings:
        # 目标串同样翻转，这样从左向右的指针，就等价于别人的从右向左匹配！
        s = s[::-1]  
        s_len = len(s)
        
        s_idx = 0
        p_idx = 0
        star_idx = -1
        s_tmp_idx = -1
        
        # 经典贪心双指针通配符匹配（O(1) 极致空间，无任何切片/函数开销）
        while s_idx < s_len:
            # 1. 字符匹配，或者遇到 '?'
            if p_idx < p_len and (p[p_idx] == '?' or p[p_idx] == s[s_idx]):
                s_idx += 1
                p_idx += 1
            # 2. 遇到 '*'，记录 '*' 的位置，并试着让它先匹配 0 个字符
            elif p_idx < p_len and p[p_idx] == '*':
                star_idx = p_idx
                p_idx += 1
                s_tmp_idx = s_idx
            # 3. 匹配失败，但前面有 '*'，回溯到 '*' 的位置，让 '*' 多吞噬一个字符
            elif star_idx != -1:
                p_idx = star_idx + 1
                s_tmp_idx += 1
                s_idx = s_tmp_idx
            # 4. 彻底匹配失败且没有 '*' 可以救场
            else:
                break
                
        # 处理模式串尾部剩余的连续 '*'
        while p_idx < p_len and p[p_idx] == '*':
            p_idx += 1
            
        # 如果两个指针都走到了尽头，说明完美匹配
        if p_idx == p_len and s_idx == s_len:
            out.append("YES")
        else:
            out.append("NO")

    # 一次性高速打印，杜绝 I/O 卡顿
    sys.stdout.write('\n'.join(out) + '\n')

if __name__ == '__main__':
    solve()


## G 汉诺塔

In [ ]:
import sys

def solve():
    # 极速读取所有输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    n = int(input_data[0])
    priorities = input_data[1:7]

    # 将字母 A, B, C 映射为数字 0, 1, 2 方便计算
    peg_map = {'A': 0, 'B': 1, 'C': 2}
    
    # dest[i][u] 表示高度为 i 的塔从 u 出发最终会落在哪个柱子上
    dest = [[0] * 3 for _ in range(n + 1)]
    # cost[i][u] 表示对应的移动步数
    cost = [[0] * 3 for _ in range(n + 1)]

    # 1. 初始状态：只有 1 个盘子的情况
    for u in range(3):
        char_u = 'ABC'[u]
        # 在优先级列表中找到第一个合法的起始动作
        for op in priorities:
            if op[0] == char_u:
                v = peg_map[op[1]]
                dest[1][u] = v
                cost[1][u] = 1
                break
    
    # 2. 状态转移：从 2 个盘子一直推导到 n 个盘子
    for i in range(2, n + 1):
        for u in range(3):
            v = dest[i-1][u]           # i-1 塔第一次降落的柱子
            w = 3 - u - v              # 剩下的那根空柱子 (利用 0+1+2=3 的数学性质)
            x = dest[i-1][v]           # i-1 塔第二次降落的柱子
            
            # 分支 A：3 步直接合体
            if x == w:
                dest[i][u] = w
                cost[i][u] = cost[i-1][u] + 1 + cost[i-1][v]
            # 分支 B：5 步折返合体
            else: 
                dest[i][u] = v
                cost[i][u] = cost[i-1][u] + 1 + cost[i-1][v] + 1 + cost[i-1][u]

    # 题目要求的是把所有盘子从 A(即 0) 移走的步数
    print(cost[n][0])

if __name__ == '__main__':
    solve()


## H 马步距离

In [ ]:
import sys

def solve():
    # 极速读取四个坐标
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    xp = int(input_data[0])
    yp = int(input_data[1])
    xs = int(input_data[2])
    ys = int(input_data[3])
    
    # 利用对称性，全部转化为从 (0,0) 到第一象限的 (x, y) 的最短距离
    # 并且保证 x 永远是较长的那条边
    x = abs(xp - xs)
    y = abs(yp - ys)
    if x < y:
        x, y = y, x
        
    # 两个极近距离的硬核死角特判
    if x == 1 and y == 0:
        print(3)
        return
    if x == 2 and y == 2:
        print(4)
        return
        
    # O(1) 数学核心公式
    # Python 中 向上取整的除法 ⌈a/b⌉ 可以优雅地写成 (a + b - 1) // b
    ans = max((x + 1) // 2, (x + y + 2) // 3)
    
    # 奇偶性同调修正：步数的奇偶性必须跟目标坐标和的奇偶性完全一致
    if ans % 2 != (x + y) % 2:
        ans += 1
        
    print(ans)

if __name__ == '__main__':
    solve()


## I 直方图最大矩形

In [ ]:
#
# 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
#
# 
# @param heights int整型一维数组 
# @return int整型
#
class Solution:
    def largestRectangleArea(self , heights: list[int]) -> int:
        # 哨兵技巧 1：在末尾追加高度为 0 的虚拟柱子，迫使循环结束时弹出栈内所有元素
        # 使用 + [0] 隐式创建新列表，防止直接污染原始参数
        heights = heights + [0]
        
        # 哨兵技巧 2：栈初始放入 -1 作为最左侧的虚拟边界
        stack = [-1]
        max_area = 0
        
        for i in range(len(heights)):
            # 当遇到比栈顶更矮的柱子时，说明栈顶柱子的“右边界”确定了
            while stack[-1] != -1 and heights[i] < heights[stack[-1]]:
                # 弹出栈顶柱子，并以此高度计算面积
                h = heights[stack.pop()]
                
                # 此时新的栈顶就是原栈顶的“左边界”
                # 宽度 = 右边界(i) - 左边界(stack[-1]) - 1
                w = i - stack[-1] - 1
                
                area = h * w
                if area > max_area:
                    max_area = area
                    
            # 当前柱子入栈，维持单调递增
            stack.append(i)
            
        return max_area


## J 消防局的设立

In [ ]:
import sys

def solve():
    # 极速读取整个输入流，规避一切多余空格或换行带来的脏数据干扰
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    n = int(input_data[0])
    
    # 只有一个基地时，只需建 1 个消防局
    if n == 1:
        print(1)
        return
        
    # 构造父节点数组，a[i] 表示节点 i 的父节点
    # 为了让索引对其，我们在开头垫两个 0：a[2] 就对应 input_data[1]
    a = [0, 0] + [int(x) for x in input_data[1:n]]
    
    ans = 0
    INF = 10**9
    
    # d1[i]: 子树中距离 i 最近的消防局距离
    d1 = [INF] * (n + 1)
    # d2[i]: 子树中距离 i 最远的未覆盖基地的距离
    d2 = [-INF] * (n + 1)
    
    # 因为 a[i] < i 恒成立，所以直接从大到小倒序遍历，就是完美的自底向上拓扑序
    for i in range(n, 0, -1):
        
        # 1. 如果当前节点 i 还没有被子树中的消防局覆盖，那它自己就成了一个距离为 0 的未覆盖点
        if d1[i] > 2:
            if d2[i] < 0:
                d2[i] = 0
                
        # 2. 如果子树中最远的未覆盖点距离当前节点 i 已经达到 2，
        # 则不能再往上拖了，必须立刻在节点 i 建立一个消防局！
        if d2[i] == 2:
            ans += 1
            d1[i] = 0
            d2[i] = -INF
            
        # 3. 如果当前节点不是根节点，将其状态上报传递给父节点
        if i > 1:
            p = a[i]
            if d1[i] + 1 < d1[p]:
                d1[p] = d1[i] + 1
            if d2[i] + 1 > d2[p]:
                d2[p] = d2[i] + 1
                
            # 神仙判断：如果父节点目前收集到的最近消防局，能够顺手覆盖掉最远的未覆盖点
            # 那么这个未覆盖点就被消灭了，重置 d2[p]
            if d1[p] + d2[p] <= 2:
                d2[p] = -INF
                
    # 遍历结束后，如果根节点（1号基地）周围还有未被覆盖的基地（距离为0或1）
    # 那么只能在根节点再补建一个消防局了
    if d2[1] >= 0:
        ans += 1
        
    # 输出极简的结果
    print(ans)

if __name__ == '__main__':
    solve()
